In [1]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/pretrain/finish/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())
model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))

/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


<All keys matched successfully>

In [2]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(device=device)

In [3]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=256,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.2
)

prompt = [
    "<s><user>请问你是由谁训练研发的呢？</s>\n<s><bot>",
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>能否介绍一下中国的首都呢？</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>"
]
input_ids = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

In [4]:
with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>请问你是由谁训练研发的呢？</s>
<s><bot>: 1929年10月15日,第五届世界互联网大会在浙江乌镇开幕.1949年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1949年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1958年10月15日,第五届世界互联网大会在浙江乌镇开幕.1964年1月15日,第五届世界互联网大会在浙江乌镇开幕.
 1964年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1964年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1974年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1984年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1984年10月15日,第五届世界互联网大会在浙江乌镇开幕.
 1997年10月15日,第五届世界互联网大会
1: <pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>的"河洛"(The Mirage of Venus)是19世纪英国数学家,数学家,数学家和建筑师协会(IAEA)于1936年1月1日宣布的英国皇家学会和皇家学会为英国皇家学会(IAEA)的前身.1947年, II.1948年,II.1955年,II.1958年,II.1965年,II.1978年,II.1982年,II.1992年,II.1995年,II.1998年,II.1999年,II.2001年,II.2012年,II.2020年,II.2030年,II.2040年,II.2050年,II.2060年,II.2062年,II.2070年,II.2080年,II.2094年,II.2096年,II.2097年,II.2098年
2: <s><user>Could you please give a C++ example for quick sort?</s>
<s><bot>网科技讯 5月27日消息(记者王丽娟)中国科学院院士,中国科学院院士,中国科学院院士,中国工程院院士,国家自然科学基金委员会专家委员会副主任陈和生,1989年12月生,浙江大学数学系系主任.1994年12月生,中国科学院院士,中国科学院院士,中国科学院数学与系统科学研究院研究员,1995年12月生,中国科学院数学与系统科学研究院研究员,中

In [5]:
#使用续写形式
prompt = [
    "北京是中国的首都，",
    "There is a story for the cat and the mouse:"
]
input_ids = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad>北京是中国的首都，, 2014
2014年10月29日, 2014年10月17日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日, 2014年10月19日,
1: There is a story for the cat and the mouse: it's a story of the human race, the human mind, and the human body. The story is told in the narrative of the story. The story is told in the narrative of the story.
The story of the human soul is told in the narrative of the story. The story is told in the narrative of the story. The story is told in the narrative of the story and the story is told in the narrative of the story. The story is written in a narrative that has a beginning and a conclusion. The story is told in the narrative that is told in the narrative of the story.
The story of the human soul is told in the narrative of the story and the story is told in the narrative of the story.
The story 